# 03 - Estimer ventes et revenus

Steam ne publie pas les ventes. On les estime à partir des avis : **1 avis ≈ 20 à 60 ventes** (méthode Boxleiter), puis on retire la part de Steam (30 %).
Les fonctions sont dans `src/estimation.py`.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.cleaning import load_clean
from src.estimation import (SALES_PER_REVIEW, STEAM_SHARE, REFUND_RATE, PRICE_PAID_RATIO,
                            estimate_sales, net_revenue, add_estimates)

pd.options.display.float_format = '{:,.2f}'.format

## 1. Tester les fonctions à la main

In [2]:
for n in [0, 10, 1000]:
    print(n, 'avis ->', estimate_sales(n, 'low'), '/', estimate_sales(n), '/', estimate_sales(n, 'high'), 'ventes')

0 avis -> 0 / 0 / 0 ventes
10 avis -> 200 / 400 / 600 ventes
1000 avis -> 20000 / 40000 / 60000 ventes


In [3]:
# Check du guide : 1 000 avis, 10 $, scénario moyen, sans remboursements ni promotions.
# À la main : 1 000 x 40 = 40 000 ventes ; 40 000 x 10 $ x 0,70 = 280 000 $.
sales = estimate_sales(1000)
print(sales, 'ventes ;', net_revenue(sales, 10, refund_rate=0, price_paid_ratio=1), '$ de revenu net')

40000 ventes ; 280000.0 $ de revenu net


## 2. Appliquer au dataset

In [4]:
games = load_clean('../data/games_clean.csv')
df = add_estimates(games)
df[['name', 'price', 'reviews', 'sales_low', 'sales_mid', 'sales_high', 'revenue_mid']].head()

,name,price,reviews,sales_low,sales_mid,sales_high,revenue_mid
0,Supipara - Chapter 1 Spring Has Come!,14.97,255,5100,10200,15300,"72,147.92"
1,Mystery Solitaire The Black Raven,4.99,24,480,960,1440,"2,263.46"
2,Armored Brigade II,39.99,130,2600,5200,7800,"98,255.43"
3,Hepta Beats,0.99,32,640,1280,1920,598.75
4,MUMBA IV: Egypt Jewels,0.59,25,500,1000,1500,278.77


## 3. Hypothèses réalistes : remboursements et promotions

In [5]:
basic = add_estimates(games, refund_rate=0, price_paid_ratio=1)
print('Remboursements :', REFUND_RATE, '| prix payé / prix affiché :', PRICE_PAID_RATIO)
print('Revenu médian sans ajustement :', round(basic['revenue_mid'].median()), '$')
print('Revenu médian avec ajustements :', round(df['revenue_mid'].median()), '$')
print('Rapport :', round(df['revenue_mid'].sum() / basic['revenue_mid'].sum(), 3))

Remboursements : 0.1 | prix payé / prix affiché : 0.75
Revenu médian sans ajustement : 2505 $
Revenu médian avec ajustements : 1691 $
Rapport : 0.675


Les deux ajustements multiplient tous les revenus par 0,9 x 0,75 ≈ 0,68 : un tiers de moins.
**On garde la version ajustée** : les remboursements existent sur Steam (moins de 2 h de jeu, moins de 14 jours), et la plupart des jeux vendent une grande partie de leurs copies en soldes. Ignorer les deux surestimerait le revenu, et donc les chances de rentabilité.

## 4. Les 10 jeux au revenu estimé le plus haut

In [6]:
top10 = df.sort_values('revenue_mid', ascending=False).head(10)
top10[['name', 'release_year', 'price', 'reviews', 'sales_mid', 'revenue_mid']]

,name,release_year,price,reviews,sales_mid,revenue_mid
25854,Black Myth: Wukong,2024,59.99,1150098,46003920,"1,303,993,763.48"
66969,ELDEN RING,2022,59.98,1056677,42267080,"1,197,872,294.09"
40330,Cyberpunk 2077,2020,59.97,844921,33796840,"957,661,343.79"
3155,Rust,2018,39.98,1227784,49111360,"927,740,601.65"
34777,Baldur's Gate 3,2023,59.99,760662,30426480,"862,446,942.88"
56363,Red Dead Redemption 2,2019,59.96,733542,29341680,"831,282,070.25"
38228,HELLDIVERS™ 2,2024,39.99,1017635,40705400,"769,139,726.98"
23338,The Witcher 3: Wild Hunt,2015,39.90,835349,33413960,"629,945,034.39"
64716,DARK SOULS™ III,2016,59.99,413775,16551000,"469,142,646.52"
31917,DayZ,2018,49.98,430252,17210080,"406,425,504.74"


Le classement est crédible : ce sont des succès mondiaux connus (Black Myth: Wukong, Elden Ring, Baldur's Gate 3, Helldivers 2...). L'ordre de grandeur (plusieurs dizaines de millions de ventes) est cohérent avec les chiffres annoncés pour ces jeux.

# Étape 6 - Valider la méthode d'estimation

## 1. Comparaison avec la fourchette SteamSpy
On retire les jeux à 0 avis et ceux dont la fourchette est « 0 - 20000 » (trop imprécise).

In [7]:
valid = df[(df['reviews'] > 0) & (df['owners_min'] > 0)].copy()
print(len(valid), 'jeux avec une fourchette SteamSpy exploitable')

inside = (valid['sales_mid'] >= valid['owners_min']) & (valid['sales_mid'] <= valid['owners_max'])
below = valid['sales_mid'] < valid['owners_min']
above = valid['sales_mid'] > valid['owners_max']
print('Dans la fourchette :', round(100 * inside.mean(), 1), '%')
print('En dessous         :', round(100 * below.mean(), 1), '%')
print('Au-dessus          :', round(100 * above.mean(), 1), '%')

18950 jeux avec une fourchette SteamSpy exploitable
Dans la fourchette : 14.3 %
En dessous         : 83.1 %
Au-dessus          : 2.6 %


In [8]:
# Même calcul pour les 3 scénarios, et pour l'intervalle complet bas-haut
for scenario in ['low', 'mid', 'high']:
    col = valid[f'sales_{scenario}']
    share = ((col >= valid['owners_min']) & (col <= valid['owners_max'])).mean()
    print(scenario, ':', round(100 * share, 1), '% dans la fourchette')
overlap = (valid['sales_high'] >= valid['owners_min']) & (valid['sales_low'] <= valid['owners_max'])
print('Intervalle bas-haut qui recoupe la fourchette :', round(100 * overlap.mean(), 1), '%')

low : 3.5 % dans la fourchette
mid : 14.3 % dans la fourchette
high : 20.2 % dans la fourchette
Intervalle bas-haut qui recoupe la fourchette : 27.9 %


## 2. Calibrer : combien d'owners par avis ?

In [9]:
valid['owners_mid'] = (valid['owners_min'] + valid['owners_max']) / 2
valid['ratio'] = valid['owners_mid'] / valid['reviews']
print('Ratio médian owners / avis :', round(valid['ratio'].median(), 1))

Ratio médian owners / avis : 244.8


In [10]:
valid['price_band'] = pd.cut(valid['price'], bins=[0, 5, 10, 15, 20, 30, 1000],
                             labels=['<5$', '5-10$', '10-15$', '15-20$', '20-30$', '30$+'])
pd.options.display.float_format = '{:,.1f}'.format
print(valid.groupby('price_band', observed=True)['ratio'].agg(['median', 'count']))
print(valid.groupby('release_year')['ratio'].agg(['median', 'count']))
pd.options.display.float_format = '{:,.0f}'.format

            median  count
price_band               
<5$          810.8   8354
5-10$        265.2   4091
10-15$       141.1   2243
15-20$        98.6   2088
20-30$        74.0   1283
30$+          56.2    891
              median  count
release_year               
2015           215.5   1345
2016           241.4   1737
2017           225.1   1694
2018           200.0   1601
2019           158.0   1348
2020           250.0   1770
2021           625.0   2275
2022           472.3   2192
2023           284.6   2087
2024           222.6   2049
2025           184.9    852


## 3. Jeux connus aux ventes annoncées publiquement

In [11]:
# Le ratio dépend beaucoup de la taille du jeu : regardons-le par nombre d'avis
valid['review_band'] = pd.cut(valid['reviews'], bins=[0, 100, 1000, 10000, 10**8],
                              labels=['1-100', '101-1000', '1001-10000', '10000+'])
pd.options.display.float_format = '{:,.1f}'.format
print(valid.groupby('review_band', observed=True)['ratio'].agg(['median', 'count']))
pd.options.display.float_format = '{:,.0f}'.format

             median  count
review_band               
1-100       3,181.8   7110
101-1000      169.5   6888
1001-10000     71.9   3860
10000+         47.6   1092


**Lecture** : le ratio médian global (≈ 245 owners par avis) est très loin de 40, mais il est biaisé :
- on ne garde que les jeux à 20 000 owners ou plus : un petit jeu à 100 avis tombe forcément « en dessous » de la fourchette ;
- les owners SteamSpy comptent aussi les clés offertes, les bundles et les week-ends gratuits, surtout pour les jeux à moins de 5 $ (ratio > 800) ;
- pour les jeux mieux mesurés, le ratio tombe à 72 owners par avis (1 000 à 10 000 avis) et 48 (plus de 10 000 avis). Comme owners ≥ ventes, cela reste compatible avec 40 à 60 ventes par avis.

**Décision** : on garde 20 / 40 / 60. Notre estimation est donc plutôt **prudente** (en dessous de SteamSpy pour la plupart des jeux), ce qui est le bon sens d'erreur pour un outil qui conseille de dépenser un budget.

In [12]:
# Chiffres officiels annoncés par les studios ou leurs éditeurs.
# Attention : ce sont des ventes toutes plateformes (sauf Valheim, PC seulement en 2022),
# comparées à notre estimation Steam seule, calculée avec les avis de 2026.
known = pd.DataFrame([
    {'name': 'Valheim', 'official_sales': 10_000_000, 'date': 'avril 2022, PC',
     'source': 'https://www.pcgamer.com/valheim-has-sold-over-10-million-copies'},
    {'name': 'Balatro', 'official_sales': 5_000_000, 'date': 'janvier 2025, toutes plateformes',
     'source': 'https://www.gematsu.com/2025/01/balatro-sales-top-five-million'},
    {'name': 'Hollow Knight', 'official_sales': 15_000_000, 'date': 'août 2025, toutes plateformes',
     'source': 'https://www.avclub.com/hollow-knight-hits-all-time-peak-player-count-sells-15-million-copies'},
])
comparison = known.merge(df[['name', 'reviews', 'sales_low', 'sales_mid', 'sales_high']], on='name')
comparison['écart_mid_%'] = (100 * (comparison['sales_mid'] - comparison['official_sales'])
                             / comparison['official_sales']).round(0)
comparison[['name', 'date', 'official_sales', 'sales_low', 'sales_mid', 'sales_high', 'écart_mid_%']]

,name,date,official_sales,sales_low,sales_mid,sales_high,écart_mid_%
0,Valheim,"avril 2022, PC",10000000,9996560,19993120,29989680,100
1,Balatro,"janvier 2025, toutes plateformes",5000000,3071320,6142640,9213960,23
2,Hollow Knight,"août 2025, toutes plateformes",15000000,8318920,16637840,24956760,11


**Lecture** : pour les 3 jeux, le chiffre officiel tombe entre nos scénarios bas et moyen.
Le scénario moyen surestime (de +10 % pour Hollow Knight à +100 % pour Valheim), mais les comparaisons sont imparfaites : Valheim a continué à se vendre après 2022, et les chiffres de Balatro et Hollow Knight incluent consoles et mobiles.
Ces trois succès sont des jeux très « commentés » (beaucoup d'avis par acheteur) : pour eux, le scénario bas est le plus juste. Pour un petit jeu, l'étape précédente suggère au contraire que 40 est prudent.

**Phrase pour la note** : notre estimation moyenne tombe dans la fourchette SteamSpy pour 14 % des jeux (et sous la fourchette pour 83 %, donc prudente) ; sur 3 jeux connus, le chiffre officiel est encadré par nos scénarios bas et moyen.